In [1]:

import glob
import pandas as pd
import os
import pyarrow as pa
import numpy as np
import pyarrow.parquet as pq
import re


from scipy.stats import spearmanr, pearsonr


In [2]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types,
    explanation_m,
    linear_coders,
    explanation_k,
    explanation_lambda,
    explanation_seed
    )

/root/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/root/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

In [3]:
NUM_TEST_INSTANCES = 1000 

In [4]:
k = len(explanation_k)
m = 1 # len(explanation_m)
lambda_ = len(explanation_lambda)
explanation_seed_ = 5

In [5]:
num_explanation_types = 1 # Self
num_explanation_types += k*m # explanations.TopKMostInfluential,
num_explanation_types += k*m # explanations.TopKLeastInfluential,
num_explanation_types += k*m # explanations.TopKMostHelpful,
num_explanation_types += k*m # explanations.TopKMostHarmful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostHarmful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostHelpful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostInfluential,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationLeastInfluential,
num_explanation_types += k*m # explanations.DIVINEMostHelpful,
num_explanation_types += k*m # explanations.DIVINEMostHarmful,
num_explanation_types += k*m # explanations.DIVINEMostInfluential,
num_explanation_types += k*m # explanations.DIVINELeastInfluential,
num_explanation_types += k*m # explanations.AIDE
num_explanation_types += k*m*explanation_seed_
num_explanation_types

137

In [6]:
num_explanation_types_bm25 = 1 # Self
num_explanation_types_bm25 += k*m # explanations.TopKMostInfluential,
num_explanation_types_bm25 += k*m # explanations.TopKLeastInfluential,
num_explanation_types_bm25 += k*m*lambda_ # explanations.FacilityLocationMostInfluential,
num_explanation_types_bm25 += k*m*lambda_ # explanations.FacilityLocationLeastInfluential,
num_explanation_types_bm25 += k*m # explanations.DIVINEMostInfluential,
num_explanation_types_bm25 += k*m # explanations.DIVINELeastInfluential,
num_explanation_types_bm25 += k*m # explanations.AIDE
num_explanation_types_bm25 += k*m*explanation_seed_
num_explanation_types_bm25

81

## Scoring

In [7]:
df_scoring = pq.ParquetDataset("results/scoring").read().to_pandas()

In [8]:
group_sizes = df_scoring.groupby(
    ["model","explanation_type","train_dataset","train_split","test_dataset","test_split",
     "estimator", "linear_coder"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

Series([], dtype: int64)

In [9]:
n_settings = num_explanation_types*3*2 + num_explanation_types_bm25*3


In [10]:
len(df_scoring["explanation_type"].unique()) == num_explanation_types

True

In [11]:
assert n_settings == len(df_scoring.groupby(["explanation_type", "model", "estimator"]))
assert len(group_sizes[group_sizes != NUM_TEST_INSTANCES]) == 0

## Validation

In [12]:
df_validation = pq.ParquetDataset("results/validation").read().to_pandas()


In [13]:
len(df_validation)

1065000

In [14]:
set(df_scoring["explanation_type"].unique()) - set(df_validation["explanation_type"].unique())

set()

In [15]:
missing = set(
    df_scoring[["explanation_type", "model", "estimator"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
) - set(
    df_validation[["explanation_type", "model", "estimator"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

if missing:
    missing_validation_explanation_types, missing_validation_models, missing_validation_estimators = map(set, zip(*missing))
else:
    missing_validation_explanation_types = set()
    missing_validation_models = set()
    missing_validation_estimators = set()

print(missing_validation_explanation_types)
print(missing_validation_models)
print(missing_validation_estimators)


set()
set()
set()


In [16]:
len(df_validation["explanation_type"].unique()) == num_explanation_types

True

In [17]:
group_sizes = df_validation.groupby(
    ["model","explanation_type", "train_dataset","train_split","test_dataset","test_split",
     "estimator"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

Series([], dtype: int64)

In [18]:
 len(df_validation.groupby(["explanation_type", "model", "estimator"]))

1065

In [19]:
n_settings = num_explanation_types*3*2 + num_explanation_types_bm25*3
assert n_settings == len(df_validation.groupby(["explanation_type", "model", "estimator"]))